# BirdCLEF 2026 - Multi-Label EffNet/ConvNeXt Baseline
This notebook implements a sliding window multi-label classification pipeline for BirdCLEF 2026.

## 1. Setup & Imports

In [1]:
import os
import gc
import sys
import math
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import librosa
import soundfile as sf

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import torchaudio.transforms as T

import timm
import ast
import albumentations as A
from sklearn.metrics import average_precision_score, roc_auc_score # roc_auc_macro is the competition metric
from sklearn.model_selection import StratifiedKFold

os.environ['TORCH_HOME'] = '/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b0/1'

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'timm'

## 2. Configuration & Paths

In [11]:
class Config:
    # 1. Config Pathing (Dynamic check for Local vs Kaggle)
    import os
    if os.path.exists('/kaggle/input/competitions/birdclef-2026'):
        ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
    else:
        ROOT_DIR = '../../../../data/raw'
        
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    SOUNDSCAPE_CSV = os.path.join(ROOT_DIR, 'train_soundscapes_labels.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')
    
    # Audio Setup
    SR = 32000
    WINDOW_SECONDS = 5
    HOP_SECONDS = 2.5  # For inference overlap
    
    # Mel Spectrogram Setup
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 20
    FMAX = 16000
    
    # Training Setup
    SEED = 42
    BATCH_SIZE = 32
    EPOCHS = 20
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 4
    
    # Model Setup
    MODEL_NAME = 'tf_efficientnet_b0' 
    NUM_CLASSES = 0 
    
CFG = Config()

## 3. Utility Functions

In [4]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)

## 4. Dataset & Data Processing

In [5]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None, is_train=True):
        self.df = df
        self.audio_dir = audio_dir
        self.transform = transform
        self.is_train = is_train
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        try:
            # 1. Speed Fix: sf.info + sf.read chunking
            info = sf.info(audio_path)
            total_samples = info.frames
            
            if total_samples > self.window_samples:
                if self.is_train:
                    start = random.randint(0, total_samples - self.window_samples)
                else:
                    start = 0
                y, _ = sf.read(audio_path, start=start, frames=self.window_samples, always_2d=True)
            else:
                y, _ = sf.read(audio_path, always_2d=True)
                
            y = y.mean(axis=1) # Mono
            
            if len(y) < self.window_samples:
                pad_len = self.window_samples - len(y)
                y = np.pad(y, (0, pad_len))
        except Exception as e:
            y = np.zeros(self.window_samples)
            
        # 4. Background Noise Augmentation
        if self.is_train and random.random() < 0.5:
            noise = np.random.randn(len(y))
            y = y + 0.005 * noise
            
        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel_spec = self.mel_transform(y_tensor)
        mel_spec = self.amplitude_to_db(mel_spec)
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        image = torch.stack([mel_spec, mel_spec, mel_spec])
        
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        target[row['label_id']] = 1.0
        
        if 'secondary_labels' in row and pd.notna(row['secondary_labels']):
            
            try:
                sec_labels = ast.literal_eval(row['secondary_labels'])
                for sl in sec_labels:
                    if sl in label_to_id:
                        target[label_to_id[sl]] = 1.0
            except:
                pass
        
        return image, target

class SoundscapeDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None):
        self.df = df
        self.audio_dir = audio_dir
        self.transform = transform
        self.window_samples = CFG.SR * CFG.WINDOW_SECONDS
        
        self.mel_transform = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        
        h, m, s = map(int, row['start'].split(':'))
        start_sample = (h * 3600 + m * 60 + s) * CFG.SR
        
        try:
            y, _ = sf.read(audio_path, start=start_sample, stop=start_sample + self.window_samples, always_2d=True)
            y = y.mean(axis=1) # Mono
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except Exception as e:
            y = np.zeros(self.window_samples)
            
        y_tensor = torch.tensor(y, dtype=torch.float32)
        mel_spec = self.mel_transform(y_tensor)
        mel_spec = self.amplitude_to_db(mel_spec)
        mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
        image = torch.stack([mel_spec, mel_spec, mel_spec])
        
        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        labels = str(row['primary_label']).split(';')
        for label in labels:
            if label in label_to_id:
                target[label_to_id[label]] = 1.0
                
        return image, target

## 5. Augmentations

In [6]:
# Utilities for SpecAugment, Mixup, etc.
def get_train_transforms():
    return A.Compose([
        A.CoarseDropout(max_holes=1, max_height=16, max_width=16, p=0.5),
    ])

def mixup_data(x, y, alpha=0.2):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).cuda()
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

## 6. Model Architecture

In [7]:
class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif 'convnext' in model_name:
            in_features = self.backbone.head.fc.in_features
            self.backbone.head.fc = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out

## 7. Loss & Optimization

In [8]:
def get_optimizer(model):
    return optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

def get_scheduler(optimizer):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

def get_criterion():
    return nn.BCEWithLogitsLoss()  # Multi-label loss

## 8. Training Loops

In [9]:
def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    epoch_loss = 0
    for images, targets in tqdm(loader, desc='Train'):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        
        # Mixup
        images, targets_a, targets_b, lam = mixup_data(images, targets)
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(loader)

def valid_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    preds, true_targets = [], []
    
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Valid'):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
            
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    
    # ROC-AUC skipping classes with no TPs
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        # Only calculate AUC if there is at least one positive and one negative sample
        if len(np.unique(true_targets[:, i])) > 1:
            auc = roc_auc_score(true_targets[:, i], preds[:, i])
            auc_scores.append(auc)
            
    final_score = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_score

## 9. Main Execution

In [12]:
# Load and Prepare Data
df = pd.read_csv(CFG.TRAIN_CSV)

# Create Label Mapping
unique_labels = sorted(df['primary_label'].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}
df['label_id'] = df['primary_label'].map(label_to_id)

CFG.NUM_CLASSES = len(unique_labels)
print(f"Detected {CFG.NUM_CLASSES} classes.")

# Load Soundscape Data for Validation
ss_df = pd.read_csv(CFG.SOUNDSCAPE_CSV)
print(f'Loaded {len(ss_df)} soundscape segments for validation.')

Detected 206 classes.


In [ ]:
# Training Loop (Balanced Dataloaders & Disjoint OOF Validation)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Fix for Rare Classes (Option 2: Duplication) ---
# Identify species with only 1 sample
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()

if rare_birds:
    print(f"Found {len(rare_birds)} species with only 1 sample. Duplicating for stratification...")
    rare_df = df[df['label_id'].isin(rare_birds)].copy()
    # Duplicate so each class has at least 2 samples for the stratified split
    df = pd.concat([df, rare_df], ignore_index=True)

# 1. Disjoint OOF Split (80% Train / 20% Valid Clips)
from sklearn.model_selection import train_test_split
train_df, valid_df_clips = train_test_split(
    df, test_size=0.2, stratify=df['label_id'], random_state=CFG.SEED
)

print(f"\n{'='*20} Balanced Training & Disjoint Validation {'='*20}")
print(f"Train Clips: {len(train_df)} | Valid Clips: {len(valid_df_clips)} | Valid SS: {len(ss_df)}")

# 2. Weighted Samplers for BOTH (Training and Validation Balance)
def get_sampler(df_subset):
    class_counts = df_subset['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / (class_counts + 1e-6)
    # Map weights to each sample
    weights = df_subset['label_id'].map(lambda x: class_weights[x] if x < len(class_weights) else 0).values
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_sampler = get_sampler(train_df)
valid_sampler = get_sampler(valid_df_clips)

# 3. Datasets & Loaders
train_ds = BirdDataset(train_df, CFG.TRAIN_AUDIO_DIR, is_train=True)
valid_ds_clips = BirdDataset(valid_df_clips, CFG.TRAIN_AUDIO_DIR, is_train=False)
valid_ds_ss = SoundscapeDataset(ss_df, CFG.SOUNDSCAPE_DIR)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=train_sampler, num_workers=CFG.NUM_WORKERS)
valid_loader_clips = DataLoader(valid_ds_clips, batch_size=CFG.BATCH_SIZE, sampler=valid_sampler, num_workers=CFG.NUM_WORKERS)
valid_loader_ss = DataLoader(valid_ds_ss, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

# 4. Initialization
model = BirdModel(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
optimizer = get_optimizer(model)
scheduler = get_scheduler(optimizer)
criterion = get_criterion()
scaler = torch.cuda.amp.GradScaler()

best_score = 0
for epoch in range(CFG.EPOCHS):
    start_time = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
    
    # Validate on BOTH disjoint clips and soundscapes
    _, val_auc_clips = valid_epoch(model, valid_loader_clips, criterion, device)
    _, val_auc_ss = valid_epoch(model, valid_loader_ss, criterion, device)
    
    scheduler.step()
    
    # Log results (Focus on Soundscape AUC as primary metric)
    duration = time.time() - start_time
    print(f"Epoch {epoch} | Loss: {train_loss:.4f} | Clip AUC: {val_auc_clips:.4f} | SS AUC: {val_auc_ss:.4f} | Time: {int(duration)}s")
    
    if val_auc_ss > best_score:
        best_score = val_auc_ss
        torch.save(model.state_dict(), "best_model.pth")
        print(f"--> Saved best model with Soundscape AUC: {best_score:.4f}")

## 10. Inference & Submission

In [ ]:
import glob
model.load_state_dict(torch.load('best_model.pth'))
model.eval()
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
if os.path.exists(TEST_DIR):
    test_files = glob.glob(f'{TEST_DIR}/*.ogg')
    print(f'Ready for inference on {len(test_files)} files.')
else:
    print('Test directory not found (Local Mode).')